## Olist E-Commerce: SQL Data Checks and ERD Preparation

## 1. Setup the notebook

This notebook uses DuckDB to inspect five Olist e-commerce tables, run data-quality checks, and support the relationship diagram.

**Notebook workflow**

1. Load the CSV files into DuckDB
2. Review each table: columns, grain, and candidate keys
3. Test table relationships using foreign-key checks
4. Find data quality issues
5. Build a star schema

**Tables:** `customers`, `orders`, `order_items`, `products`, `sellers`

In [29]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
WAREHOUSE_DIR = DATA_DIR / "warehouse"
csv = RAW_DIR / "olist_orders_dataset.csv"

In [ ]:
print(duckdb.sql(f"SELECT count(*) FROM '{csv}'"))

### DuckDB lets query CSV data with SQL. The database file persists locally so later notebooks can reuse the loaded raw tables.

In [4]:
DB_PATH = WAREHOUSE_DIR / "ecommerce.duckdb"
con = duckdb.connect(str(DB_PATH))

In [5]:
def q(sql_str: str) -> pd.DataFrame:
    """Run a SQL query in DuckDB and return a pandas DataFrame."""
    return con.sql(sql_str).df()

## 2. Load the source tables into a **raw schema**

A Schema is a namespace that groups tables. We keep the source copies in **raw**
so that they remain distinguishable from cleaned warehouse tables we will create later.

### Define a mapping to transform column types

In [ ]:
RAW_TABLES: dict[str, dict[str, str]] = {
    "customers": {"customer_zip_code_prefix": "VARCHAR"},  # keep leading zeros
    "orders": {},
    "order_items": {},
    "products": {},
    "sellers": {}
}

### Define some helper functions

In [8]:
def csv_path(table: str) -> Path:
    """"""
    return RAW_DIR / f"olist_{table}_dataset.csv"

def check_source_files(tables) -> None:
    """Fail once, list every missing file"""
    missing = [csv_path(t) for t in tables if not csv_path(t).exists()]
    if missing:
        names = "\n".join(f" - {p}" for p in missing)
        raise FileNotFoundError(f"Missing CSV files: {names}")

#### Define schema name for source data

In [ ]:
RAW_SCHEMA = "raw"

### Load data into source tables

In [11]:
def load_raw_table(con: duckdb.DuckDBPyConnection, table: str, column_types: dict[str, str]) -> int:
    target = f'{RAW_SCHEMA}."{table}"'
    source = con.read_csv(str(csv_path(table)), header=True, dtype=column_types or None)
    con.execute(f"DROP TABLE IF EXISTS {target}")
    source.to_table(target)
    return con.execute(f"SELECT count(*) FROM {target}").fetchone()[0]

### Sanity check if we have all required source files

In [12]:
check_source_files(RAW_TABLES)

### Create **RAW_SCHEMA** if it does not exist

In [13]:
con.execute(f"CREATE SCHEMA IF NOT EXISTS {RAW_SCHEMA}")

### Execute **load_raw_table** to load all source data

In [ ]:
row_counts = {t: load_raw_table(con, t, types) for t, types in RAW_TABLES.items()}

### Check count of rows inserted into each `raw`.`<table>`

In [14]:
pd.DataFrame(row_counts.items(), columns=["table_name", "row_count"])

,table_name,row_count
0,customers,99441
1,orders,99441
2,order_items,112650
3,products,32951
4,sellers,3095


## 3. Look at each table

For every table we ask: **what does one row represent?** This is the *grain*. Getting the grain right is the most important step in dimensional modeling.

In [ ]:
for t in RAW_TABLES:
    print(f"----- {t} -----")
    display(q(f"DESCRIBE raw.{t}")[["column_name", "column_type", "null"]])
    display(q(f"SELECT * FROM raw.{t} LIMIT 5"))

`SUMMARIZE`profiles every column is one command: min, max, average, null percentage,
approximate distinct count.

In [ ]:
q("SUMMARIZE raw.order_items")

In [ ]:
q("SUMMARIZE raw.order_items")

In [ ]:
q("SUMMARIZE raw.products")

### Grain of each table

- `customers`: one customer record (`customer_id`); `customer_unique_id` groups records for a real-world customer.
- `orders`: one order (`order_id`).
- `order_items`: one item line within an order; identified by the composite key (`order_id`, `order_item_id`).
- `products`: one product (`product_id`).
- `sellers`: one seller (`seller_id`).

The following checks must be interpreted using query results; do not infer keys or relationship cardinalities from names alone.


## 4. Primary Keys

A **primary key** uniquely identifies a row. A column is a valid candidate only if `count(*) = count(distinct ...)` and it has no nulls.

- A **composite key** uses several columns together (`order_items`)
- A **natural key** comes from the business (`customer_unique_id`).
- A **surrogate key** is generated by the warehouse (`customer_key = 1, 2, 3...`)

In [ ]:
q("""
SELECT 'customers.customer_id'         AS candidate, count(*) AS n, count(DISTINCT customer_id) AS n_distinct FROM raw.customers
UNION ALL SELECT 'customers.customer_unique_id',         count(*), count(DISTINCT customer_unique_id) FROM raw.customers
UNION ALL SELECT 'orders.order_id',                      count(*), count(DISTINCT order_id) FROM raw.orders
UNION ALL SELECT 'order_items.order_id (alone)',         count(*), count(DISTINCT order_id) FROM raw.order_items
UNION ALL SELECT 'order_items(order_id, order_item_id)', count(*), count(DISTINCT (order_id, order_item_id)) FROM raw.order_items
UNION ALL SELECT 'products.product_id',                  count(*), count(DISTINCT product_id) FROM raw.products
UNION ALL SELECT 'sellers.seller_id',                    count(*), count(DISTINCT seller_id) FROM raw.sellers
""")

### Primary-key null checks

Uniqueness counts alone are not enough: every proposed key must also be checked for null values. For `order_items`, both `order_id` and `order_item_id` form the composite key.


In [ ]:
q("""
SELECT 'customers.customer_id' AS candidate, COUNT(*) FILTER (WHERE customer_id IS NULL) AS null_rows FROM raw.customers
UNION ALL SELECT 'orders.order_id', COUNT(*) FILTER (WHERE order_id IS NULL) FROM raw.orders
UNION ALL SELECT 'order_items.order_id', COUNT(*) FILTER (WHERE order_id IS NULL) FROM raw.order_items
UNION ALL SELECT 'order_items.order_item_id', COUNT(*) FILTER (WHERE order_item_id IS NULL) FROM raw.order_items
UNION ALL SELECT 'products.product_id', COUNT(*) FILTER (WHERE product_id IS NULL) FROM raw.products
UNION ALL SELECT 'sellers.seller_id', COUNT(*) FILTER (WHERE seller_id IS NULL) FROM raw.sellers;
""")


## 4. Foreign key relationships

```
customers 1 ---- N orders 1 ---- N order_items N ---- 1 products
```

A foreign key is healthy when every value exists in the parent table. An **orphan** is
a child row with no parent.

In [ ]:
q("""
SELECT 'orders without customer' AS check_name, count(*) AS n
  FROM raw.orders o ANTI JOIN raw.customers c USING (customer_id)
UNION ALL SELECT 'items without order', count(*)
  FROM raw.order_items i ANTI JOIN raw.orders o USING (order_id)
UNION ALL SELECT 'items without product', count(*)
  FROM raw.order_items i ANTI JOIN raw.products p USING (product_id)
UNION ALL SELECT 'items without seller', count(*)
    FROM raw.order_items i ANTI JOIN raw.sellers s USING (seller_id)
UNION ALL SELECT 'orders without items', count(*)
  FROM raw.orders o ANTI JOIN raw.order_items i USING (order_id)
""")

### Measure relationship cardinalities

This query reports the minimum and maximum number of child rows per parent in the observed data. A zero minimum means some parents have no children; a minimum of one means every parent has at least one child in this dataset.


In [ ]:
q("""
WITH child_counts AS (
  SELECT 'customers -> orders' AS relationship, c.customer_id AS parent_id, COUNT(o.order_id) AS child_count
  FROM raw.customers c LEFT JOIN raw.orders o ON o.customer_id = c.customer_id
  GROUP BY c.customer_id
  UNION ALL
  SELECT 'orders -> order_items', o.order_id, COUNT(i.order_id)
  FROM raw.orders o LEFT JOIN raw.order_items i ON i.order_id = o.order_id
  GROUP BY o.order_id
  UNION ALL
  SELECT 'products -> order_items', p.product_id, COUNT(i.product_id)
  FROM raw.products p LEFT JOIN raw.order_items i ON i.product_id = p.product_id
  GROUP BY p.product_id
  UNION ALL
  SELECT 'sellers -> order_items', s.seller_id, COUNT(i.seller_id)
  FROM raw.sellers s LEFT JOIN raw.order_items i ON i.seller_id = s.seller_id
  GROUP BY s.seller_id
)
SELECT relationship, MIN(child_count) AS min_children,
       MAX(child_count) AS max_children,
       COUNT(*) FILTER (WHERE child_count = 0) AS parents_with_no_children
FROM child_counts
GROUP BY relationship
ORDER BY relationship;
""")


No orphans in the child-to-parent direction. But 775 orders have no items. An **INNER JOIN** from orders
to items would silently remove them.

## 5. Data Quality

we check five dimensions: completeness(nulls), uniqueness, validity (values in a sensible range),
consistency (fields agree), timeliness (coverage)

### 5.1 Orders with no items

#### Exercise 1

In [ ]:
q("""
SELECT o.order_status, COUNT(*) AS orders_without_items
FROM raw.orders o
ANTI JOIN raw.order_items i USING (order_id)
GROUP BY o.order_status
ORDER BY orders_without_items DESC;
""")

### 5.2 Check order status and class imbalance

#### Exercise 2

In [ ]:
q("""
SELECT order_status,
       COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM raw.orders
GROUP BY order_status
ORDER BY n DESC;
""")

### 5.3 Missing and inconsistent timestamps

#### Exercise 3

In [ ]:
q("""
SELECT
  COUNT(*) FILTER (WHERE order_approved_at IS NULL) AS approved_null,
  COUNT(*) FILTER (WHERE order_delivered_carrier_date IS NULL) AS carrier_null,
  COUNT(*) FILTER (WHERE order_delivered_customer_date IS NULL) AS delivered_null,
  COUNT(*) FILTER (WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL) AS delivered_but_no_date,
  COUNT(*) FILTER (WHERE order_delivered_carrier_date > order_delivered_customer_date) AS carrier_after_customer
FROM raw.orders;
""")

### 5.4 Time coverage

#### Exercise 4

In [ ]:
monthly = q("""
SELECT DATE_TRUNC('month', CAST(order_purchase_timestamp AS TIMESTAMP)) AS month,
       COUNT(*) AS orders
FROM raw.orders
WHERE order_purchase_timestamp IS NOT NULL
GROUP BY 1
ORDER BY 1;
""")

### 5.5 Product completeness check

#### Exercise 5

In [ ]:
q("""
SELECT
  COUNT(*) AS total_products,
  COUNT(*) FILTER (WHERE product_category_name IS NULL OR TRIM(product_category_name) = '') AS missing_category,
  COUNT(*) FILTER (WHERE product_name_lenght IS NULL) AS missing_name_length,
  COUNT(*) FILTER (WHERE product_description_lenght IS NULL) AS missing_description_length,
  COUNT(*) FILTER (WHERE product_photos_qty IS NULL) AS missing_photo_count,
  COUNT(*) FILTER (WHERE product_weight_g IS NULL) AS missing_weight,
  COUNT(*) FILTER (WHERE product_length_cm IS NULL OR product_height_cm IS NULL OR product_width_cm IS NULL) AS missing_dimensions
FROM raw.products;
""")